In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('Data/encode_train.csv')
test = pd.read_csv('Data/encode_test.csv')
valid = pd.read_csv('Data/encode_valid.csv')

In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.sparse import issparse
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.svm import OneClassSVM
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
# from category_encoders import BinaryEncoder
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from utils import evaluate_outlier_model
# PYOD pacakages
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.qmcd import QMCD
from pyod.models.gmm import GMM
from pyod.models.cd import CD
from pyod.models.loda import LODA
from pyod.models.inne import INNE


In [4]:
bool_col = [
    'FirstTimeHomebuyerFlag',
    'SuperConformingFlag',
    'CreditScore_MissFLag',
    'OriginalDTI_MissFLag',
    'HighRiskCredit',
    'HighLTV',
    'HighDTI',
    'HighInterestRate',
    'LTV_MissFlag'
]

cat_col = [
    'NumberOfUnits',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'ProgramIndicator',
    'PropertyValMethod',
    'BalloonIndicator'
]

num_col = [
    'CreditScore',
    'MI_Pct',
    'OriginalDTI',
    'OriginalUPB',
    'OriginalLTV',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    '0_1_UPB_Diff',
    '1_2_UPB_Diff',
    '2_3_UPB_Diff',
    '3_4_UPB_Diff',
    '4_5_UPB_Diff',
    '5_6_UPB_Diff',
    '6_7_UPB_Diff',
    '7_8_UPB_Diff',
    '8_9_UPB_Diff',
    '9_10_UPB_Diff',
    '10_11_UPB_Diff',
    '11_12_UPB_Diff',
    '12_13_UPB_Diff',
    'longest_unchanged_UPB',
    'avg_repayment_ratio',
    'pct_months_late',
    'Principal_reduction_rate',
    'Amortization_slope',
    'UPB_rebound_count',
    'UPB_autocorr',
    'UPB_skew',
    'UPB_kurtosis',
    'LTV_mean',
    'LTV_std',
    'LTV_slope',
    'LTV_rebound_count',
    'Interest_rate_std',
    'Interest_rate_slope',
    'Early_repayment_ratio',
    'CreditScore_LTV_Ratio',
    'CreditScore_DTI_Ratio',
    'LTV_DTI_Product',
    'DebtServiceRatio',
    'CompositeRiskScore',
    'OriginalLTV_delta',
    'MSA_freq_enc',
    'PropertyState_freq_enc',
    'SellerName_freq_enc',
    'ServicerName_freq_enc'
]

In [5]:
import copy
all_feature_dict = {}
for value in bool_col:
    all_feature_dict[value] = "bool"
for value in cat_col:
    all_feature_dict[value] = "cat"
for value in num_col:
    all_feature_dict[value] = "num"
all_feature_dict_copy = copy.deepcopy(all_feature_dict)

In [7]:
X_train = train[cat_col + num_col + bool_col]
X_valid = valid[cat_col + num_col + bool_col]
X_test = test.drop(columns="Id")

y_valid = valid[['index', 'target']]

In [9]:
for col in cat_col:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)
    X_test[col] = X_test[col].astype(str)
for col in bool_col:
    X_train[col] = X_train[col].astype(int)
    X_valid[col] = X_valid[col].astype(int)
    X_test[col] = X_test[col].astype(int)

C:\Users\zheng\AppData\Local\Temp\ipykernel_16232\362698848.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].astype(str)
C:\Users\zheng\AppData\Local\Temp\ipykernel_16232\362698848.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_valid[col] = X_valid[col].astype(str)
C:\Users\zheng\AppData\Local\Temp\ipykernel_16232\362698848.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

# Greedy
Greedy search for each unsupervised methods

## OneClassSVM greedy search

In [21]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = OneClassSVM(
            kernel="rbf",     # RBF works well for non-linear anomalies
            nu=0.05,          # Upper bound on fraction of anomalies in training set
            gamma="scale"     # Kernel coefficient (can tune)
        )
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5054
AP=0.1338, AUC=0.5293
AP=0.1357, AUC=0.5300
AP=0.1685, AUC=0.5953
AP=0.1993, AUC=0.6063
AP=0.2205, AUC=0.6163
AP=0.2373, AUC=0.6122
AP=0.2379, AUC=0.6273
AP=0.2984, AUC=0.5973
Round 1: Added 'pct_months_late' -> AP=0.2984
AP=0.3863, AUC=0.7226
Round 2: Added 'NumberOfUnits' -> AP=0.3863
AP=0.3866, AUC=0.7155
AP=0.3951, AUC=0.7201
AP=0.4023, AUC=0.7214
Round 3: Added 'FirstTimeHomebuyerFlag' -> AP=0.4023

No improvement found. Stopping at 3 features.


## Isolation Forest

In [24]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1283, AUC=0.5054
AP=0.1301, AUC=0.5108
AP=0.1329, AUC=0.5280
AP=0.1474, AUC=0.5712
AP=0.2106, AUC=0.6446
AP=0.3456, AUC=0.6352
Round 1: Added 'pct_months_late' -> AP=0.3456
AP=0.3485, AUC=0.6577
AP=0.3719, AUC=0.6827
AP=0.3794, AUC=0.6655
Round 2: Added 'UPB_rebound_count' -> AP=0.3794
AP=0.3809, AUC=0.6889
Round 3: Added 'HighDTI' -> AP=0.3809
AP=0.3846, AUC=0.6948
AP=0.3860, AUC=0.6969
AP=0.3891, AUC=0.6951
Round 4: Added 'HighRiskCredit' -> AP=0.3891

No improvement found. Stopping at 4 features.


## LOF

In [23]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = LocalOutlierFactor(n_neighbors=8, novelty=True, leaf_size = 15)
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1261, AUC=0.5000
AP=0.1263, AUC=0.5007
AP=0.1274, AUC=0.5007
AP=0.1280, AUC=0.5026
AP=0.1286, AUC=0.4950
AP=0.1358, AUC=0.4952
AP=0.1454, AUC=0.5111
AP=0.1919, AUC=0.5377
Round 1: Added 'pct_months_late' -> AP=0.1919
AP=0.1930, AUC=0.5379
AP=0.1945, AUC=0.5390
AP=0.2180, AUC=0.5551
AP=0.2231, AUC=0.5352
AP=0.2327, AUC=0.5409
AP=0.2342, AUC=0.5443
AP=0.2551, AUC=0.5525
Round 2: Added '7_8_UPB_Diff' -> AP=0.2551
AP=0.2554, AUC=0.5516
AP=0.2562, AUC=0.5607
AP=0.2743, AUC=0.5729
Round 3: Added '0_1_UPB_Diff' -> AP=0.2743
AP=0.2743, AUC=0.5729
Round 4: Added 'CreditScore_MissFLag' -> AP=0.2743
AP=0.2743, AUC=0.5729
Round 5: Added 'OriginalDTI_MissFLag' -> AP=0.2743

No improvement found. Stopping at 5 features.


# PYOD 
## ECOD

In [25]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = ECOD()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5054
AP=0.1301, AUC=0.5108
AP=0.1329, AUC=0.5280
AP=0.1474, AUC=0.5712
AP=0.1745, AUC=0.5924
AP=0.1889, AUC=0.6077
AP=0.2089, AUC=0.6267
AP=0.2353, AUC=0.6343
AP=0.3538, AUC=0.6569
Round 1: Added 'pct_months_late' -> AP=0.3538
AP=0.3559, AUC=0.6542
AP=0.3815, AUC=0.6792
Round 2: Added 'HighDTI' -> AP=0.3815
AP=0.3960, AUC=0.6868
Round 3: Added 'HighInterestRate' -> AP=0.3960

No improvement found. Stopping at 3 features.


## COPOD

In [26]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = COPOD()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5054
AP=0.1301, AUC=0.5108
AP=0.1329, AUC=0.5280
AP=0.1474, AUC=0.5712
AP=0.2323, AUC=0.6548
Round 1: Added 'CreditScore' -> AP=0.2323
AP=0.2426, AUC=0.6821
AP=0.2723, AUC=0.6676
AP=0.2737, AUC=0.6590
AP=0.2853, AUC=0.6825
Round 2: Added 'Early_repayment_ratio' -> AP=0.2853
AP=0.2946, AUC=0.7015
AP=0.3056, AUC=0.6794
AP=0.3073, AUC=0.6707
AP=0.3178, AUC=0.6873
Round 3: Added 'UPB_rebound_count' -> AP=0.3178
AP=0.3181, AUC=0.6911
AP=0.3269, AUC=0.7034
AP=0.3294, AUC=0.6850
AP=0.3435, AUC=0.6738
Round 4: Added 'pct_months_late' -> AP=0.3435
AP=0.3524, AUC=0.6897
Round 5: Added 'HighDTI' -> AP=0.3524
AP=0.3598, AUC=0.6978
AP=0.3603, AUC=0.6948
Round 6: Added 'LTV_rebound_count' -> AP=0.3603
AP=0.3658, AUC=0.7025
Round 7: Added 'HighInterestRate' -> AP=0.3658

No improvement found. Stopping at 7 features.


## QMCD

In [29]:
from pyod.models.qmcd import QMCD

selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = QMCD()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:107: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew = stats.skew(scores)
C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:108: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurt = stats.kurtosis(scores)


AP=0.1261, AUC=0.5000


C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:107: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew = stats.skew(scores)
C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:108: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurt = stats.kurtosis(scores)
C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:107: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew = stats.skew(scores)
C:\Users\zheng\anaconda3\envs\IS5126\Lib\site-packages\pyod\models\qmcd.py:108: RuntimeWarning: Precision loss occurred in moment calculation due to catastroph

AP=0.2375, AUC=0.6559
Round 1: Added 'CreditScore' -> AP=0.2375
AP=0.3364, AUC=0.6801
Round 2: Added 'pct_months_late' -> AP=0.3364
AP=0.3567, AUC=0.6769
Round 3: Added 'UPB_rebound_count' -> AP=0.3567
AP=0.3738, AUC=0.6838
Round 4: Added 'Early_repayment_ratio' -> AP=0.3738
AP=0.3752, AUC=0.6921
Round 5: Added 'CompositeRiskScore' -> AP=0.3752
AP=0.3761, AUC=0.6970
Round 6: Added 'ServicerName_freq_enc' -> AP=0.3761

No improvement found. Stopping at 6 features.


## GMM

In [40]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = GMM()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5054
AP=0.1301, AUC=0.5108
AP=0.1329, AUC=0.5280
AP=0.1474, AUC=0.5712
AP=0.2155, AUC=0.6038
AP=0.2180, AUC=0.6053
AP=0.2253, AUC=0.6255
AP=0.2319, AUC=0.6083
AP=0.3067, AUC=0.6510
Round 1: Added 'pct_months_late' -> AP=0.3067
AP=0.3110, AUC=0.6752
AP=0.3182, AUC=0.6595
AP=0.3322, AUC=0.6651
AP=0.3426, AUC=0.6957
Round 2: Added 'longest_unchanged_UPB' -> AP=0.3426
AP=0.3440, AUC=0.6917
AP=0.3518, AUC=0.7089
AP=0.3557, AUC=0.7035
AP=0.3765, AUC=0.7051
Round 3: Added 'CreditScore' -> AP=0.3765
AP=0.3765, AUC=0.7051
AP=0.3792, AUC=0.7061
AP=0.3825, AUC=0.7202
AP=0.3933, AUC=0.7143
Round 4: Added 'UPB_rebound_count' -> AP=0.3933
AP=0.3933, AUC=0.7143
AP=0.3951, AUC=0.7180
AP=0.3985, AUC=0.7272
Round 5: Added 'HighDTI' -> AP=0.3985
AP=0.3997, AUC=0.7293
AP=0.4004, AUC=0.7363
Round 6: Added 'HighInterestRate' -> AP=0.4004
AP=0.4004, AUC=0.7363
AP=0.4012, AUC=0.7385
AP=0.4015, AUC=0.7378
Round 7: Added 'MI_Pct' -> AP=0.4015

No improvement found. Stopping at 7 features.


## CD

In [38]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        try:
            model  = CD()
            # print(feature)
            type_col = all_feature_dict_copy[feature]
            # print(type_col)
            X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
            X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
            # print(X_train_current.head())
            if type_col == "cat":
                ap, roc_auc, scores = evaluate_outlier_model(
                    model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
                )
            elif type_col =="bool":
                ap, roc_auc, scores = evaluate_outlier_model(
                    model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
                )
            else:
                ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
                )
            if ap > best_round_ap:
                    print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                    best_round_ap = ap
                    best_feature = feature
        except:
            continue
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5053
AP=0.1301, AUC=0.5108
AP=0.1358, AUC=0.5300
Round 1: Added 'LoanPurpose' -> AP=0.1358
AP=0.1392, AUC=0.5418
AP=0.1416, AUC=0.5536
AP=0.1521, AUC=0.5847
AP=0.1528, AUC=0.5716
AP=0.1622, AUC=0.5831
AP=0.1626, AUC=0.5853
AP=0.1685, AUC=0.5946
AP=0.1759, AUC=0.6130
AP=0.2085, AUC=0.6181
Round 2: Added 'pct_months_late' -> AP=0.2085
AP=0.2192, AUC=0.6183
AP=0.2246, AUC=0.6206
AP=0.2247, AUC=0.6471
AP=0.2315, AUC=0.6471
AP=0.2540, AUC=0.6407
Round 3: Added 'UPB_rebound_count' -> AP=0.2540
AP=0.2542, AUC=0.6596
Round 4: Added 'longest_unchanged_UPB' -> AP=0.2542
AP=0.2581, AUC=0.6556
AP=0.2600, AUC=0.6753
AP=0.2703, AUC=0.6833
AP=0.2812, AUC=0.6846
Round 5: Added 'OriginalLTV_delta' -> AP=0.2812
AP=0.2881, AUC=0.6855
Round 6: Added 'FirstTimeHomebuyerFlag' -> AP=0.2881
AP=0.2883, AUC=0.6955
AP=0.2947, AUC=0.6980
Round 7: Added 'HighInterestRate' -> AP=0.2947

No improvement found. Stopping at 7 features.


## LODA

In [13]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = LODA()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1277, AUC=0.5054
AP=0.1301, AUC=0.5108
AP=0.1329, AUC=0.5280
AP=0.1331, AUC=0.5275
AP=0.1474, AUC=0.5712
AP=0.2310, AUC=0.6661
AP=0.2590, AUC=0.6563
Round 1: Added 'pct_months_late' -> AP=0.2590
AP=0.3283, AUC=0.6335
AP=0.3746, AUC=0.6600
Round 2: Added 'OccupancyStatus' -> AP=0.3746
AP=0.3861, AUC=0.6938
Round 3: Added '8_9_UPB_Diff' -> AP=0.3861
AP=0.3966, AUC=0.6752
Round 4: Added 'OriginalInterestRate' -> AP=0.3966

No improvement found. Stopping at 4 features.


## INNE

In [14]:
selected_bool, selected_cat, selected_num = [], [], []
remaining = cat_col + bool_col + num_col
best_ap = 0
for round_num in range(25):
    best_feature= None
    best_round_ap = best_ap
    
    for feature in remaining:
        model  = INNE()
        # print(feature)
        type_col = all_feature_dict_copy[feature]
        X_train_current = X_train[selected_bool + selected_cat + selected_num + [feature]]
        X_valid_current = X_valid[selected_bool + selected_cat + selected_num + [feature]]
        # print(X_train_current.head())
        if type_col == "cat":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat+[feature], selected_num, selected_bool, sparse_output=False, ispyod=True
            )
        elif type_col =="bool":
            ap, roc_auc, scores = evaluate_outlier_model(
                model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num, selected_bool+[feature],  sparse_output=False, ispyod=True
            )
        else:
            ap, roc_auc, scores = evaluate_outlier_model(
            model, X_train_current, X_valid_current, y_valid["target"], selected_cat, selected_num+[feature], selected_bool,  sparse_output=False, ispyod=True
            )
        if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
    if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected_bool) + len(selected_cat)+ len(selected_num)} features.")
            break
    type_col = all_feature_dict_copy[best_feature]
    if type_col == "cat":
        selected_cat.append(best_feature)
    elif type_col =="bool":
        selected_bool.append(best_feature)
    else:
        selected_num.append(best_feature)
    remaining.remove(best_feature)
    best_ap = best_round_ap
    
    print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")

AP=0.1275, AUC=0.5054
AP=0.1313, AUC=0.5173
AP=0.1329, AUC=0.5280
AP=0.1474, AUC=0.5712
AP=0.2116, AUC=0.6409
AP=0.2128, AUC=0.6157
AP=0.2463, AUC=0.6310
AP=0.3538, AUC=0.6569
Round 1: Added 'pct_months_late' -> AP=0.3538
AP=0.3667, AUC=0.6402
AP=0.3711, AUC=0.6680
AP=0.3903, AUC=0.6974
Round 2: Added 'OriginalUPB' -> AP=0.3903
AP=0.4000, AUC=0.7003
Round 3: Added 'SuperConformingFlag' -> AP=0.4000

No improvement found. Stopping at 3 features.
